# 🚀 Week 5 Part 2: Model Deployment with FastAPI & Docker

## Overview
Knowing how to deploy models is a key differentiator in ML interviews.

## 🎯 Learning Objectives
1. Build a FastAPI prediction endpoint
2. Containerize with Docker
3. Write basic API documentation
4. Handle errors gracefully

## 💡 Interview Insight
"How would you take a model from notebook to production?" is a common question.

---

## 1. Saving the Model for Deployment

In [ ]:
# ============================================================
# TRAIN AND SAVE A MODEL
# ============================================================

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Load and train
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Create pipeline (preprocessing + model)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print("Model Performance:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Save model and metadata
model_dir = Path('model_artifacts')
model_dir.mkdir(exist_ok=True)

# Save pipeline
joblib.dump(pipeline, model_dir / 'iris_model.joblib')

# Save metadata
metadata = {
    'feature_names': iris.feature_names,
    'target_names': iris.target_names.tolist(),
    'model_type': 'RandomForestClassifier',
    'version': '1.0.0'
}
joblib.dump(metadata, model_dir / 'metadata.joblib')

print(f"\n✅ Model saved to {model_dir}")

---
## 2. FastAPI Application

### The FastAPI code structure

In [ ]:
# ============================================================
# FASTAPI APPLICATION CODE (app.py)
# ============================================================
# Save this as app.py to run the API

fastapi_code = '''
"""
Iris Classification API
========================
A simple ML model serving API using FastAPI.

Run with: uvicorn app:app --reload
Docs at: http://localhost:8000/docs
"""

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import joblib
import numpy as np
from pathlib import Path

# Initialize FastAPI app
app = FastAPI(
    title="Iris Classification API",
    description="A machine learning API for iris flower classification",
    version="1.0.0"
)

# Load model and metadata
MODEL_PATH = Path("model_artifacts/iris_model.joblib")
METADATA_PATH = Path("model_artifacts/metadata.joblib")

try:
    model = joblib.load(MODEL_PATH)
    metadata = joblib.load(METADATA_PATH)
except FileNotFoundError:
    model = None
    metadata = None


# Request/Response schemas
class IrisFeatures(BaseModel):
    """Input features for iris classification."""
    sepal_length: float = Field(..., ge=0, le=10, description="Sepal length in cm")
    sepal_width: float = Field(..., ge=0, le=10, description="Sepal width in cm")
    petal_length: float = Field(..., ge=0, le=10, description="Petal length in cm")
    petal_width: float = Field(..., ge=0, le=10, description="Petal width in cm")
    
    class Config:
        json_schema_extra = {
            "example": {
                "sepal_length": 5.1,
                "sepal_width": 3.5,
                "petal_length": 1.4,
                "petal_width": 0.2
            }
        }


class PredictionResponse(BaseModel):
    """Prediction response with class and probabilities."""
    predicted_class: str
    predicted_class_id: int
    probabilities: dict
    model_version: str


class BatchPredictionRequest(BaseModel):
    """Batch prediction request."""
    instances: List[IrisFeatures]


class HealthResponse(BaseModel):
    """Health check response."""
    status: str
    model_loaded: bool
    version: str


# Endpoints
@app.get("/", response_model=dict)
async def root():
    """Root endpoint with API info."""
    return {
        "message": "Iris Classification API",
        "docs": "/docs",
        "health": "/health"
    }


@app.get("/health", response_model=HealthResponse)
async def health_check():
    """Health check endpoint for monitoring."""
    return HealthResponse(
        status="healthy" if model is not None else "unhealthy",
        model_loaded=model is not None,
        version=metadata.get("version", "unknown") if metadata else "unknown"
    )


@app.post("/predict", response_model=PredictionResponse)
async def predict(features: IrisFeatures):
    """Make a single prediction."""
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    # Prepare features
    X = np.array([[features.sepal_length, features.sepal_width,
                   features.petal_length, features.petal_width]])
    
    # Predict
    prediction = model.predict(X)[0]
    probabilities = model.predict_proba(X)[0]
    
    # Format response
    return PredictionResponse(
        predicted_class=metadata["target_names"][prediction],
        predicted_class_id=int(prediction),
        probabilities={
            name: float(prob) 
            for name, prob in zip(metadata["target_names"], probabilities)
        },
        model_version=metadata["version"]
    )


@app.post("/predict/batch")
async def predict_batch(request: BatchPredictionRequest):
    """Make batch predictions."""
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    # Prepare features
    X = np.array([[f.sepal_length, f.sepal_width, f.petal_length, f.petal_width]
                  for f in request.instances])
    
    # Predict
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)
    
    # Format response
    results = []
    for pred, prob in zip(predictions, probabilities):
        results.append({
            "predicted_class": metadata["target_names"][pred],
            "predicted_class_id": int(pred),
            "probabilities": {
                name: float(p) for name, p in zip(metadata["target_names"], prob)
            }
        })
    
    return {"predictions": results, "count": len(results)}
'''

# Save the FastAPI app code
with open('app.py', 'w') as f:
    f.write(fastapi_code)

print("✅ FastAPI app saved to app.py")
print("\n📋 Key components:")
print("  - Pydantic models for request/response validation")
print("  - Health check endpoint for monitoring")
print("  - Single and batch prediction endpoints")
print("  - Auto-generated docs at /docs")

---
## 3. Docker Configuration

In [ ]:
# ============================================================
# DOCKERFILE
# ============================================================

dockerfile_content = '''# Use official Python runtime as base
FROM python:3.10-slim

# Set working directory
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    gcc \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first (for caching)
COPY requirements.txt .

# Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app.py .
COPY model_artifacts/ ./model_artifacts/

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Run the application
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)

print("✅ Dockerfile created")

In [ ]:
# ============================================================
# REQUIREMENTS FILE
# ============================================================

requirements_content = '''fastapi>=0.100.0
uvicorn[standard]>=0.22.0
scikit-learn>=1.3.0
numpy>=1.24.0
joblib>=1.3.0
pydantic>=2.0.0
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("✅ requirements.txt created")

In [ ]:
# ============================================================
# DOCKER COMPOSE FILE
# ============================================================

docker_compose = '''version: "3.8"

services:
  iris-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/model_artifacts/iris_model.joblib
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped
'''

with open('docker-compose.yml', 'w') as f:
    f.write(docker_compose)

print("✅ docker-compose.yml created")

---
## 4. Testing the API

In [ ]:
# ============================================================
# TEST SCRIPT FOR API
# ============================================================

test_script = '''#!/usr/bin/env python
"""
API Test Script
===============
Run this after starting the API with: uvicorn app:app --reload
"""

import requests
import json

BASE_URL = "http://localhost:8000"

def test_health():
    """Test health endpoint."""
    response = requests.get(f"{BASE_URL}/health")
    print("Health Check:")
    print(f"  Status: {response.status_code}")
    print(f"  Response: {response.json()}")
    return response.status_code == 200

def test_single_prediction():
    """Test single prediction."""
    data = {
        "sepal_length": 5.1,
        "sepal_width": 3.5,
        "petal_length": 1.4,
        "petal_width": 0.2
    }
    
    response = requests.post(
        f"{BASE_URL}/predict",
        json=data
    )
    
    print("\nSingle Prediction:")
    print(f"  Input: {data}")
    print(f"  Status: {response.status_code}")
    print(f"  Response: {json.dumps(response.json(), indent=2)}")
    return response.status_code == 200

def test_batch_prediction():
    """Test batch prediction."""
    data = {
        "instances": [
            {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2},
            {"sepal_length": 6.2, "sepal_width": 2.8, "petal_length": 4.8, "petal_width": 1.8},
            {"sepal_length": 7.7, "sepal_width": 3.0, "petal_length": 6.1, "petal_width": 2.3}
        ]
    }
    
    response = requests.post(
        f"{BASE_URL}/predict/batch",
        json=data
    )
    
    print("\nBatch Prediction:")
    print(f"  Status: {response.status_code}")
    print(f"  Response: {json.dumps(response.json(), indent=2)}")
    return response.status_code == 200

def test_invalid_input():
    """Test error handling."""
    data = {
        "sepal_length": -1,  # Invalid: negative value
        "sepal_width": 3.5,
        "petal_length": 1.4,
        "petal_width": 0.2
    }
    
    response = requests.post(
        f"{BASE_URL}/predict",
        json=data
    )
    
    print("\nInvalid Input Test:")
    print(f"  Input: {data}")
    print(f"  Status: {response.status_code}")
    print(f"  Expected: 422 (Validation Error)")
    return response.status_code == 422


if __name__ == "__main__":
    print("="*60)
    print("API Test Suite")
    print("="*60)
    
    tests = [
        ("Health Check", test_health),
        ("Single Prediction", test_single_prediction),
        ("Batch Prediction", test_batch_prediction),
        ("Invalid Input", test_invalid_input),
    ]
    
    results = []
    for name, test_func in tests:
        try:
            passed = test_func()
            results.append((name, passed))
        except Exception as e:
            print(f"\n{name}: ERROR - {e}")
            results.append((name, False))
    
    print("\n" + "="*60)
    print("Test Summary:")
    print("="*60)
    for name, passed in results:
        status = "✅ PASSED" if passed else "❌ FAILED"
        print(f"  {name}: {status}")
'''

with open('test_api.py', 'w') as f:
    f.write(test_script)

print("✅ test_api.py created")

---
## 5. Deployment Commands Cheat Sheet

In [ ]:
# ============================================================
# DEPLOYMENT COMMANDS
# ============================================================

print("📋 Deployment Commands Cheat Sheet")
print("="*60)

print("""
🔧 LOCAL DEVELOPMENT:
----------------------
# Run API locally (without Docker)
uvicorn app:app --reload

# Test the API
python test_api.py

# View docs
open http://localhost:8000/docs


�� DOCKER COMMANDS:
-------------------
# Build image
docker build -t iris-api .

# Run container
docker run -p 8000:8000 iris-api

# Run with docker-compose
docker-compose up -d

# View logs
docker-compose logs -f

# Stop
docker-compose down


☁️ CLOUD DEPLOYMENT (AWS Example):
-----------------------------------
# Push to ECR
aws ecr get-login-password | docker login --username AWS --password-stdin <account>.dkr.ecr.<region>.amazonaws.com
docker tag iris-api:latest <account>.dkr.ecr.<region>.amazonaws.com/iris-api:latest
docker push <account>.dkr.ecr.<region>.amazonaws.com/iris-api:latest

# Deploy to ECS/EKS/Lambda as needed
""")

---
## 6. Interview Discussion Points

### Q1: How do you handle model versioning?
**Answer:**
- Version models with semantic versioning (v1.0.0)
- Store metadata alongside model (features, metrics)
- Use MLflow or DVC for model registry
- Include version in API response

### Q2: How do you handle model updates without downtime?
**Answer:**
- Blue-green deployment
- Canary releases (route % of traffic to new model)
- Load models dynamically from S3/GCS
- Use Kubernetes rolling updates

### Q3: What about monitoring in production?
**Answer:**
- **Latency:** Track p50, p95, p99 response times
- **Throughput:** Requests per second
- **Errors:** 4xx, 5xx rates
- **Data drift:** Compare input distributions to training
- **Model drift:** Track prediction distribution changes

### Q4: How do you secure the API?
**Answer:**
- API keys or OAuth2
- Rate limiting
- Input validation (Pydantic does this)
- HTTPS only
- Logging for audit trails

---
## ✅ Week 5 Part 2 Checklist

- [x] Saved model with joblib
- [x] Created FastAPI application
- [x] Added request/response validation
- [x] Created Dockerfile
- [x] Created docker-compose.yml
- [x] Created test script
- [x] Documented deployment commands

---

**Next: Week 6 - Debugging & Reasoning** 🚀